# 03b: Assess BOM/NCI irradiance coverage

Inventory the BOM table, map substation coordinates to the nearest BOM grid point, and measure monthly coverage.

In [ ]:
from __future__ import annotations
import calendar, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p/'src'/'dnsp_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the dnsp_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT/'src'))
from dnsp_analysis.analysis_cohort import site_eligibility_path
from dnsp_analysis.config import load_config
from dnsp_analysis.db import connect
from dnsp_analysis.irradiance_coverage import (
    GeographicBounds, bom_inventory_sql, bom_locations_sql, bom_mapped_coverage_sql,
    nearest_grid_mapping, irradiance_location_map_path, irradiance_coverage_path,
    write_frame_parquet,
)
from dnsp_analysis.schemas import sql_string
pd.set_option('display.max_columns', 100); plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
CONFIG_PATH = PROJECT_ROOT/'analysis.toml'
config = load_config(CONFIG_PATH, check_inputs=False)
con = connect(config)
eligibility = con.execute(f'SELECT * FROM read_parquet({sql_string(site_eligibility_path(config))})').fetchdf()
eligible = eligibility.loc[eligibility.eligible_for_irradiance_assessment].copy()
assert not eligible.empty, 'Run notebook 03a first.'
MONTHS = [(2024,m) for m in range(8,13)] + [(2025,m) for m in range(1,8)]
bounds = GeographicBounds.from_sites(eligible, padding_degrees=0.1)
print('Eligible sites:', len(eligible)); print('BOM query bounds:', bounds)

## Athena connection and query preview

reuses the existing `/shared/aws_config.py` helper. 

Need to run `aws sso login --profile ciccada` first.

In [ ]:
BMS_REVIEW_ROOT = PROJECT_ROOT.parent/'bms_sa_review'
AWS_SHARED = BMS_REVIEW_ROOT/'shared'
assert AWS_SHARED.is_dir(), f'Expected existing AWS helper at {AWS_SHARED}'
sys.path.insert(0, str(AWS_SHARED))
from aws_config import aq
inventory_query = bom_inventory_sql(bounds, MONTHS)
locations_query = bom_locations_sql(bounds, MONTHS)
print(inventory_query)

In [ ]:
BOM_QUERY_CONFIRMATION = 'RUN BOM COVERAGE AUDIT'  # Change to: RUN BOM COVERAGE AUDIT
assert BOM_QUERY_CONFIRMATION == 'RUN BOM COVERAGE AUDIT', 'Athena queries remain locked.'

In [ ]:
inventory = aq(inventory_query, database='bom_nci')
grid_locations = aq(locations_query, database='bom_nci')
display(inventory); display(grid_locations.head())
assert set(MONTHS).issubset(set(zip(inventory.year.astype(int), inventory.month.astype(int)))), 'One or more telemetry months are absent from BOM inventory.'

## Spatial mapping

Dataset contains substation coordinates, not customer coordinates. Distance therefore measures substation-to-BOM-grid separation and does not validate site location.

In [ ]:
location_map = nearest_grid_mapping(eligible[['serial','sub_lat','sub_long']], grid_locations)
display(location_map.spatial_mapping_quality.value_counts().rename_axis('quality').reset_index(name='n_sites'))
display(location_map.distance_km.describe(percentiles=[.5,.9,.95,.99]))
fig, ax = plt.subplots(figsize=(8,4)); location_map.distance_km.hist(ax=ax, bins=30)
ax.axvline(5, color='green', linestyle='--'); ax.axvline(20, color='red', linestyle='--')
ax.set(xlabel='Nearest BOM grid distance (km)', ylabel='Sites'); plt.show()
assert not location_map.spatial_mapping_quality.eq('poor').any(), 'Poor spatial matches require review.'
write_frame_parquet(config, location_map, irradiance_location_map_path(config), overwrite=True)

## Monthly coverage at mapped grid points

In [ ]:
'''points = list(location_map[['bom_latitude','bom_longitude']].drop_duplicates().itertuples(index=False, name=None))
coverage_parts = []
for start in range(0, len(points), 40):
    query = bom_mapped_coverage_sql(points[start:start+40], MONTHS)
    coverage_parts.append(aq(query, database='bom_nci'))
coverage = pd.concat(coverage_parts, ignore_index=True)
coverage['expected_10min_timestamps'] = coverage.apply(lambda r: calendar.monthrange(int(r.year), int(r.month))[1] * 24 * 6, axis=1)
coverage['timestamp_coverage'] = coverage.n_timestamps / coverage.expected_10min_timestamps
coverage['ghi_null_fraction'] = coverage.null_ghi / coverage.n_rows
display(coverage[['timestamp_coverage','ghi_null_fraction','minimum_quality_mask','maximum_quality_mask','n_quality_mask_values']].describe(percentiles=[.01,.05,.5,.95,.99]))
display(coverage.groupby(['minimum_quality_mask','maximum_quality_mask'], dropna=False).agg(grid_months=('n_timestamps','size'), rows=('n_rows','sum'), mask_1_rows=('quality_mask_1_rows','sum')).reset_index())
display(coverage.nsmallest(20, 'timestamp_coverage'))
write_frame_parquet(config, coverage, irradiance_coverage_path(config), overwrite=True)'''

In [ ]:
points = list(
    location_map[
        ["bom_latitude", "bom_longitude"]
    ].drop_duplicates().itertuples(index=False, name=None)
)

coverage_parts = []
for start in range(0, len(points), 40):
    query = bom_mapped_coverage_sql(
        points[start : start + 40],
        MONTHS,
    )
    coverage_parts.append(aq(query, database="bom_nci"))

coverage = pd.concat(coverage_parts, ignore_index=True)

# The BOM ten-minute irradiance product contains daylight observations only.
# This fraction is descriptive of day length; it is not a completeness metric.
coverage["all_day_10min_slots"] = coverage.apply(
    lambda row: (
        calendar.monthrange(int(row.year), int(row.month))[1]
        * 24
        * 6
    ),
    axis=1,
)
coverage["all_day_slot_fraction"] = (
    coverage["n_timestamps"] / coverage["all_day_10min_slots"]
)

# Compare grid points within the same month. This allows daylight duration to
# vary seasonally while detecting points with materially fewer observations.
coverage["month_reference_daylight_timestamps"] = (
    coverage.groupby(["year", "month"])["n_timestamps"].transform("max")
)
coverage["relative_daylight_coverage"] = (
    coverage["n_timestamps"]
    / coverage["month_reference_daylight_timestamps"]
)
coverage["duplicate_rows"] = (
    coverage["n_rows"] - coverage["n_timestamps"]
)
coverage["ghi_null_fraction"] = (
    coverage["null_ghi"] / coverage["n_rows"]
)

display(
    coverage[
        [
            "all_day_slot_fraction",
            "relative_daylight_coverage",
            "ghi_null_fraction",
            "minimum_quality_mask",
            "maximum_quality_mask",
            "n_quality_mask_values",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)

display(
    coverage.groupby(
        ["minimum_quality_mask", "maximum_quality_mask"],
        dropna=False,
    )
    .agg(
        grid_months=("n_timestamps", "size"),
        rows=("n_rows", "sum"),
        mask_1_rows=("quality_mask_1_rows", "sum"),
    )
    .reset_index()
)

display(coverage.nsmallest(20, "relative_daylight_coverage"))

write_frame_parquet(
    config,
    coverage,
    irradiance_coverage_path(config),
    overwrite=True,
)

## Decision gate

Passing this gate only authorises decomposition experiments. It does not make the BOM series site-level irradiance or make net-meter power an inverter measurement.

In [ ]:
# Change only after reviewing every table above to:
# ACCEPT IRRADIANCE MAPPING FOR DECOMPOSITION EXPERIMENTS
IRRADIANCE_GATE_DECISION = "ACCEPT IRRADIANCE MAPPING FOR DECOMPOSITION EXPERIMENTS"


assert (
    IRRADIANCE_GATE_DECISION
    == "ACCEPT IRRADIANCE MAPPING FOR DECOMPOSITION EXPERIMENTS"
)

expected_grid_months = len(points) * len(MONTHS)
assert len(coverage) == expected_grid_months, (
    "One or more mapped grid-months are absent."
)
assert not coverage.duplicated(
    ["latitude", "longitude", "year", "month"]
).any()
assert coverage["duplicate_rows"].sum() == 0, (
    "Duplicate BOM rows exist within a grid point/timestamp."
)
assert coverage["relative_daylight_coverage"].min() >= 0.98, (
    "A mapped grid-month has low coverage relative to other mapped "
    "grid points in that month."
)
assert coverage["ghi_null_fraction"].max() <= 0.05, (
    "At least one mapped grid-month has more than 5% null GHI."
)

con.close()
print(
    "Irradiance coverage gate passed; "
    "decomposition experiments may be designed."
)